In [1]:
import pandas as pd
import numpy as np
import torch
import os
import sys
from tqdm import tqdm, trange
import time
import copy
os.chdir("src")

print("Current working directory:", os.getcwd())

from biked_commons.design_evaluation.design_evaluation import *
from biked_commons.conditioning import conditioning
from biked_commons.resource_utils import split_datasets_path

# os.chdir("src")
# print("Current working directory:", os.getcwd())

import argparse
import torch
import numpy as np
from matplotlib import pyplot as plt
from importlib import invalidate_caches
invalidate_caches()

from biked_commons.benchmark_models.libmoon.solver.gradient import MGDASolver, GradAggSolver, EPOSolver, MOOSVGDSolver, GradHVSolver, PMTLSolver
from biked_commons.benchmark_models.libmoon.util_global.constant import problem_dict
from biked_commons.benchmark_models.libmoon.util_global.weight_factor.funs import uniform_pref
from biked_commons.benchmark_models.libmoon.visulization.view_res import vedio_res
from biked_commons.benchmark_models.libmoon.problem.mop import mop

import sys
print(sys.version)

Current working directory: /mnt/c/Users/fabie/Desktop/biked-commons/src
3.11.11 (main, Apr 25 2025, 14:53:05) [GCC 13.3.0]


In [2]:
data = pd.read_csv(split_datasets_path("bike_bench.csv"), index_col=0)

# #sample 100
# data = data.sample(100, random_state=0)
# data_tens = torch.tensor(data.values, dtype=torch.float32)

len(data.columns)

99

In [3]:
# num_data = 1
# rider_condition = conditioning.sample_riders(num_data, split="test")
# use_case_condition = conditioning.sample_use_case(num_data, split="test")
# text_condition = conditioning.sample_text(num_data, split="test")
# condition = {
#     "Rider": torch.cat([rider_condition] * 10, dim=0),
#     "Use Case": torch.cat([use_case_condition] * 10, dim=0),
#     "Text": text_condition * 10
# }

In [4]:
class bicycle(mop):
    def __init__( self, n_var=data.shape[1], n_obj=1, lbound=-np.zeros(data.shape[1]), ubound=np.ones(data.shape[1]), condition=None, eval_fn=None):
        super().__init__(n_var=n_var,
                         n_obj=n_obj,
                         lbound=lbound,
                         ubound=ubound, )
        self.problem_name = 'bicycle'
        self.condition = condition
        self.evaluator_fn=eval_fn

    def _evaluate_torch(self, x): # All scores should be minimised
        return  self.evaluator_fn(x, self.condition)

In [5]:
from numpy import array


def test_solely_usability(args, condition, x0, prefs):
    print("Testing solely usability")
    usableEV = [
        UsabilityEvaluator(),
    ]
    evaluator_usable, _, _ = construct_tensor_evaluator(usableEV, data.columns)

    def eval_fn(x, condition): # All scores should be minimised
        return 1- evaluator_usable(x, condition)

    solver = GradAggSolver(args.step_size, args.iter, args.tol)
    problem = bicycle(condition=condition, eval_fn=eval_fn)

    res = solver.solve(problem, x=x0, prefs=prefs, args=args, ref_point=array([ 1.0 ]))

    ####### Checking improvements #######

    # Should only have affacted 3 parameters
    diff = res['x'][0] - x0.numpy()
    print('Number of parameters affacted', np.count_nonzero(diff))
    initial_usability = evaluator_usable(x0, condition)
    final_usability = evaluator_usable(torch.tensor(res['x']), condition)  

    print('Usability score initial: ', initial_usability)
    print('Usability score final: ', final_usability)

    if initial_usability < final_usability:
        print("Usability score improved")
    else:
        print("Usability score not improved")


def test_solely_aero(args, condition, x0, prefs):
    print("Testing solely aero")
    aeroEV = [
        AeroEvaluator(),
    ]
    evaluator, _, _ = construct_tensor_evaluator(aeroEV, data.columns)

    def eval_fn(x, condition): # All scores should be minimised
        return evaluator(x, condition)


    solver = GradAggSolver(args.step_size, args.iter, args.tol)
    problem = bicycle(condition=condition, eval_fn=eval_fn)

    res = solver.solve(problem, x=x0, prefs=prefs, args=args, ref_point=array([ 25.0 ]))

    ####### Checking improvements #######

    diff = res['x'][0] - x0.numpy()
    print('Number of parameters affacted', np.count_nonzero(diff))
    initial = evaluator(x0, condition)
    final = evaluator(torch.tensor(res['x']), condition) 

    print('Score initial: ', initial)
    print('Score final: ', final)

    if initial > final:
        print("Score improved")
    else:
        print("Score not improved")


def test_usability_aero(args, condition, x0, prefs, solver):

    print("Testing combined usability and aero")
    aeroEV = [
        AeroEvaluator(),
    ]
    usableEV = [
        UsabilityEvaluator(),
    ]
    evaluator_aero, _, _ = construct_tensor_evaluator(aeroEV, data.columns)
    evaluator_usable, _, _ = construct_tensor_evaluator(usableEV, data.columns)

    def eval_fn(x, condition): # All scores should be minimised
        usable_score = 1 - evaluator_usable(x, condition)
        aero_score = evaluator_aero(x, condition)
        comb = torch.cat((usable_score, aero_score), dim=1)
        return comb


    problem = bicycle(condition=condition, eval_fn=eval_fn)

    start_time = time.time()
    res = solver.solve(problem, x=x0, prefs=prefs, args=args, ref_point=array([1.0 , 25.0 ]))
    end_time = time.time()
    print(f"Time completed: {end_time - start_time:.2f} seconds")

    ####### Checking improvements #######
    x0_np = x0.numpy() if isinstance(x0, torch.Tensor) else x0
    res_x_np = res['x'].numpy() if isinstance(res['x'], torch.Tensor) else res['x']

    assert x0_np.shape == res_x_np.shape, "The shapes of x0 and res['x'] must match"

    num_affected = 0
    for i in range(x0_np.shape[0]):  # Iterate over rows
        diff = np.count_nonzero(res_x_np[i] != x0_np[i])  
        num_affected += diff  

    print('Average Number of parameters affected:', num_affected/x0.shape[0])

    condition_test = {
        "Rider": condition["Rider"][0:1],
        "Use Case": condition["Use Case"][0:1], 
        "Text": condition["Text"][0:1]
    }   

    aero_improved = 0
    usability_improved = 0

    aero_valid_counter = 0
    usability_valid_counter = 0
    for x0_row, res_row in zip(x0, res['x']):
        initial_aero = evaluator_aero(x0_row.unsqueeze(0), condition_test)
        final_aero = evaluator_aero(torch.tensor(res_row).unsqueeze(0), condition_test)  
        initial_usability = evaluator_usable(x0_row.unsqueeze(0), condition_test)
        final_usability = evaluator_usable(torch.tensor(res_row).unsqueeze(0), condition_test)  

        if not torch.isnan(initial_aero).any():
            aero_valid_counter += 1
            if initial_aero > final_aero :
                aero_improved += 1

        if not torch.isnan(initial_usability).any():
            usability_valid_counter += 1    
            if initial_usability < final_usability :
                usability_improved += 1

    if aero_valid_counter == 0:
        print("No valid aero scores found")
    else:
        print('Aero score improved: ', aero_improved/aero_valid_counter)

    if usability_valid_counter == 0:
        print("No valid usability scores found")
    else:
        print('Usability score improved: ', usability_improved/usability_valid_counter)
    return res

In [14]:
parser = argparse.ArgumentParser(description='example')
parser.add_argument('--n-partition', type=int, default=10)
parser.add_argument('--agg', type=str, default='tche')
parser.add_argument('--solver', type=str, default='agg')
parser.add_argument('--problem-name', type=str, default='VLMOP2')
parser.add_argument('--iter', type=int, default=1000)
parser.add_argument('--step-size', type=float, default=1e-2)
parser.add_argument('--tol', type=float, default=1e-6)

args = parser.parse_args(args=[
    '--n-partition', '100',
    '--agg', 'tche',
    '--solver', 'agg',
    '--iter', '1000',
    '--step-size', '0.01',
    '--tol', '1e-6'
])
args.n_var = data.shape[1]


num_data = 1
rider_condition = conditioning.sample_riders(num_data, split="test")
use_case_condition = conditioning.sample_use_case(num_data, split="test")
text_condition = conditioning.sample_text(num_data, split="test")

############### Single Objective #######################
num_objectives = 1
prefs = uniform_pref(args.n_partition, num_objectives, clip_eps=1e-2)  # Single Objective
args.n_prob = len(prefs)
condition = {
    "Rider": torch.cat([rider_condition] * prefs.shape[0], dim=0),
    "Use Case": torch.cat([use_case_condition] * prefs.shape[0], dim=0),
    "Text": text_condition * prefs.shape[0]
}

# # Initialize the initial solution 
x0 = torch.rand(args.n_prob, data.shape[1])

test_solely_usability(args, condition, x0, prefs)

test_solely_aero(args, condition, x0, prefs)

############### Multi Objective #######################

num_objectives = 2
args.n_obj = num_objectives
prefs = uniform_pref(args.n_partition, num_objectives, clip_eps=1e-2)  # Two Objective
args.n_prob = len(prefs)
condition = {
    "Rider": torch.cat([rider_condition] * prefs.shape[0], dim=0),
    "Use Case": torch.cat([use_case_condition] * prefs.shape[0], dim=0),
    "Text": text_condition * prefs.shape[0]
}
prefs = uniform_pref(args.n_partition, num_objectives, clip_eps=1e-2)  # Two Objective

x0 = torch.rand(args.n_prob, data.shape[1])

solver = GradAggSolver(args.step_size, args.iter, args.tol)

res = test_usability_aero(args, condition, x0, prefs, solver)



Testing solely usability


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:03<00:00, 285.81it/s]


Number of parameters affacted 3
Usability score initial:  tensor([[0.0066]], grad_fn=<CopySlices>)
Usability score final:  tensor([[0.5308]], grad_fn=<CopySlices>)
Usability score improved
Testing solely aero


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:08<00:00, 123.83it/s]


Number of parameters affacted 7
Score initial:  tensor([[7.5629]], grad_fn=<CopySlices>)
Score final:  tensor([[nan]], grad_fn=<CopySlices>)
Score not improved
Testing combined usability and aero


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:13<00:00, 76.42it/s]


Time completed: 13.09 seconds
Average Number of parameters affected: 7.74
Aero score improved:  0.6521739130434783
Usability score improved:  0.3695652173913043


In [15]:
################ Multiple Solver Tests #######################
solver_list = [ 'mgda', 'agg', 'moosvgd', 'hvgrad']
# 'mgda', 'agg', 'epo', 'moosvgd',
# 'hvgrad', 'pmtl'

num_objectives = 2
args.n_obj = num_objectives
prefs = uniform_pref(args.n_partition, num_objectives, clip_eps=1e-2)  # Two Objective
args.n_prob = len(prefs)
condition = {
    "Rider": torch.cat([rider_condition] * prefs.shape[0], dim=0),
    "Use Case": torch.cat([use_case_condition] * prefs.shape[0], dim=0),
    "Text": text_condition * prefs.shape[0]
}
prefs = uniform_pref(args.n_partition, num_objectives, clip_eps=1e-2)  # Two Objective


aeroEV = [
    AeroEvaluator(),
]
condition_test = {
    "Rider": condition["Rider"][0:1],
    "Use Case": condition["Use Case"][0:1], 
    "Text": condition["Text"][0:1]
}   
evaluator_aero, _, _ = construct_tensor_evaluator(aeroEV, data.columns)
x0_list = []  # Store valid rows

while len(x0_list) < args.n_prob:
    candidate_row = torch.rand(1, data.shape[1])  # Generate a candidate row (batch size 1)
    aero_score = evaluator_aero(candidate_row, condition_test)
    if not torch.isnan(aero_score).any():
        x0_list.append(candidate_row)

# Stack all valid rows into a tensor
x0 = torch.cat(x0_list, dim=0)  # Make sure input is all valid


for solver_name in solver_list:
    print(f"Testing solver: {solver_name}")
    if solver_name == 'mgda':                                          # Working well
        solver = MGDASolver(args.step_size, args.iter, args.tol)
    elif solver_name == 'agg':
        solver = GradAggSolver(args.step_size, args.iter, args.tol)
    elif solver_name == 'epo':
        solver = EPOSolver(args.step_size, args.iter, args.tol)        # Sensitive to NaN values (aero breaks it)
    elif solver_name == 'moosvgd':
        solver = MOOSVGDSolver(args.step_size, args.iter, args.tol)       #(Affecting all parameters for some reason)
    elif solver_name == 'hvgrad':
        solver = GradHVSolver(args.step_size, args.iter, args.tol)    # Only supports 2 objectives
    elif solver_name == 'pmtl':
        solver = PMTLSolver(args.step_size, args.iter, args.tol)     # Only supports 2 objectives  (Not working)
    else:
        raise Exception('solver not supported')

    test_usability_aero(
    copy.deepcopy(args),          
    {k: (v.clone() if isinstance(v, torch.Tensor) else copy.deepcopy(v)) for k, v in condition.items()},
    x0.clone(),                   
    prefs.copy(),                
    solver 
)


tensor([[nan]], grad_fn=<CopySlices>)
tensor([[10.7484]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[8.2980]], grad_fn=<CopySlices>)
tensor([[4.8326]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[10.5674]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[8.4518]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[nan]], grad_fn=<CopySlices>)
tensor([[-2.1069]], grad_fn=<CopySlices>)
tensor([[8.3319]], grad_fn=<CopySlices>)
tensor([[9.1146]], grad_fn=<CopySlices>)
tensor([[12.8880]], grad_fn=<CopySlices>)
tensor([[nan]], gra

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [10:16<00:00,  1.62it/s]


Time completed: 617.00 seconds
Average Number of parameters affected: 7.83
Aero score improved:  0.7
Usability score improved:  0.61
Testing solver: agg
Testing combined usability and aero


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:07<00:00, 132.97it/s]


Time completed: 7.52 seconds
Average Number of parameters affected: 7.2
Aero score improved:  0.5656565656565656
Usability score improved:  0.4
Testing solver: moosvgd
Testing combined usability and aero


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [08:48<00:00,  1.89it/s]


Time completed: 528.14 seconds
Average Number of parameters affected: 99.0
Aero score improved:  0.0
Usability score improved:  0.0
Testing solver: hvgrad
Testing combined usability and aero


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:30<00:00, 33.05it/s]


Time completed: 30.27 seconds
Average Number of parameters affected: 0.12
Aero score improved:  0.04040404040404041
Usability score improved:  0.02
